 # PolicyProof AI — Stage 3: AlignTune QLoRA T4

Research-only integration test. Run in a **new Google Colab T4 notebook**. Use your downloaded CuratorKIT ZIP, which contains only fictional training rows. The previous 48-case benchmark must never enter training. **Installation and GPU training default to OFF.** This is not a verified model improvement. Tiny Aya's CC-BY-NC license applies.

## 1. Check T4 and pull versioned project code.

In [ ]:
import torch, subprocess, pathlib, os, sys, json, time
assert torch.cuda.is_available(), "Select a T4 GPU under Runtime settings and connect."
print("GPU:", torch.cuda.get_device_name(0))
ROOT=pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
REV=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
print("Code revision:",REV)


## 2. Upload the CuratorKIT training ZIP and verify its integrity before extraction. Only allowlisted files are extracted.

In [ ]:
from google.colab import files
from zipfile import ZipFile
from scripts.verify_curator_archive import check_zip
uploaded=files.upload()
assert len(uploaded)==1, "Upload only the policyproof_curatorkit_train_v1.zip"
zip_name=next(iter(uploaded))
assert zip_name.endswith(".zip")
report=check_zip(zip_name)
assert report["status"]=="pass" and report["row_count"]==78
print(json.dumps(report,indent=2,ensure_ascii=False))
CURATED=ROOT/"artifacts"/"verified_curator_train"
CURATED.mkdir(parents=True,exist_ok=True)
allowed={"sft_alpaca.jsonl","sft_sharegpt.jsonl","manifest.json",
         "lexsi_provenance.json","checksums.txt","README.md","dataset_card.md","rejected.jsonl"}
with ZipFile(zip_name) as z:
    for name in sorted(allowed.intersection(z.namelist())):
        (CURATED/name).write_bytes(z.read(name))


## 3. Authenticate through the Hugging Face login UI and record the actual model SHA. Do not paste tokens into notebook cells.

In [ ]:
from huggingface_hub import whoami,notebook_login,model_info
try:
    account=whoami()
except Exception:
    notebook_login()
    account=whoami()
MODEL_ID="CohereLabs/tiny-aya-global"
MODEL_SHA=model_info(MODEL_ID).sha
assert MODEL_SHA
print("Account:",account["name"],"model SHA:",MODEL_SHA)


## 4. Explicit package-install opt-in. The official AlignTune installation is heavy and may conflict with Colab. Never install into your successful baseline runtime.

In [ ]:
INSTALL_ALIGNTUNE=False
if INSTALL_ALIGNTUNE:
    subprocess.run([sys.executable,"-m","pip","install",
        "aligntune>=0.1.10,<1","huggingface-hub>=1.31,<2"],check=True)
    subprocess.run([sys.executable,"-m","pip","check"],check=True)
    from importlib.metadata import version
    print("AlignTune:",version("aligntune"))
else:
    print("Heavyweight installation disabled")


## 5. Confirm data shape and AlignTune factory before training.

In [ ]:
from policyproof.core import read_jsonl,sha256
train_path=CURATED/"sft_alpaca.jsonl"
rows=read_jsonl(train_path)
assert len(rows)==78 and all(set(r)=={"instruction","input","output"} for r in rows)
print("Verified rows:",len(rows),"SHA256:",sha256(train_path))
if INSTALL_ALIGNTUNE:
    from aligntune.core.backend_factory import create_sft_trainer
    print("Trainer factory imported")
else:
    print("Trainer factory not checked yet")


## 6. EXPERIMENTAL QLoRA smoke training. This cell does nothing unless both flags are enabled. It may fail on the Tiny Aya architecture or T4. Do not infer any quality improvement from training completion.

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    assert INSTALL_ALIGNTUNE,"Install and validate AlignTune first."
    from datasets import load_dataset
    from aligntune.core.backend_factory import create_sft_trainer
    loaded=load_dataset("json",data_files={"train":str(train_path)},split="train")
    assert len(loaded)==78
    OUTPUT=ROOT/"artifacts"/"align_qlora_smoke"
    trainer=create_sft_trainer(
        model_name=MODEL_ID,
        dataset_name=str(CURATED),
        config_name="sft_alpaca",
        backend="trl",
        task_type="instruction_following",
        split="train",
        output_dir=str(OUTPUT),
        num_epochs=1,
        max_steps=10,
        batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_seq_length=1024,
        instruction_column="instruction",
        input_column="input",
        response_column="output",
        peft_enabled=True,
        lora_r=8,lora_alpha=16,lora_dropout=0.05,
        lora_target_modules=["q_proj","k_proj","v_proj","o_proj"],
        quantization={"load_in_4bit":True,
                      "bnb_4bit_compute_dtype":"float16",
                      "bnb_4bit_quant_type":"nf4"},
        use_gradient_checkpointing=True,
        use_flash_attention_2=False,
        attn_implementation="sdpa",
    )
    t0=time.time()
    trainer.train()
    adapter=trainer.save_model(output_dir=str(OUTPUT/"adapter"))
    RUN_INFO={"trained":True,"model":MODEL_ID,"base_sha":MODEL_SHA,
        "code_sha":REV,"train_data_sha":sha256(train_path),
        "train_rows":78,"duration_s":round(time.time()-t0,1),
        "adapter_location":str(adapter),"steps":10}
    (OUTPUT/"run_info.json").write_text(json.dumps(RUN_INFO,indent=2),encoding="utf-8")
    print("Trained and saved small integration adapter:",adapter)
else:
    print("Training OFF. No GPU training was executed.")


## 7. Export only after a successful training run. Do not label a failed or unexecuted run as successful.

In [ ]:
if RUN_TRAINING and "RUN_INFO" in globals() and RUN_INFO.get("trained"):
    import shutil
    archive=shutil.make_archive("/content/policyproof_align_smoke","zip",root_dir=OUTPUT)
    files.download(archive)
else:
    print("No trained checkpoint to download")


## Next stage

If the install fails, share the traceback without credentials. If it works, download the adapter plus run_info evidence. Then build a fresh uninspected multilingual benchmark and verify the tuned model independently. Official docs: https://aligntune.lexsi.ai/user-guide/sft/ .